# Steam Games Multiclass Classification Model

This notebook builds a multiclass classification model to predict Steam game rating groups based on metadata, pricing, platform support, and category-related features.

The target variable is derived from `positive_rate` and grouped into four rating classes:

- `75-80`
- `81-85`
- `86-90`
- `>90`

This notebook is intended as the multiclass modeling part of the Steam game success prediction project.

## 1. Import Libraries

The libraries below are used for data manipulation, modeling, evaluation, and visualization.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

## 2. Load Dataset

The dataset used here is the modeling table prepared with BigQuery SQL.

In [ ]:
drive.mount("/content/drive")

df_ml = pd.read_csv("/content/drive/MyDrive/steam_dataset_2025/steam_ml.csv")

df_ml.head()

## 3. Data Overview

Before modeling, we inspect the dataset size, column types, and missing values.

In [ ]:
print("Dataset shape:", df_ml.shape)

display(df_ml.head())
display(df_ml.info())
display(df_ml.describe(include="all").T.head(30))

In [ ]:
missing_values = (
    df_ml.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_values.head(20)

## 4. Data Filtering

Very high prices can behave as outliers and distort the model.  
The same filtering rule used in the binary model is applied here: games with `price_usd <= 100` are kept.

In [ ]:
df_ml = df_ml[df_ml["price_usd"] <= 100].copy()

print("Filtered dataset shape:", df_ml.shape)

## 5. Create Multiclass Target

Only games with a `positive_rate >= 0.75` are included in this multiclass setup.  
The target groups are created from positive review rate intervals.

In [ ]:
df_multi = df_ml[df_ml["positive_rate"] >= 0.75].copy()

bins = [0.75, 0.80, 0.85, 0.90, 1.00]
labels = ["75-80", "81-85", "86-90", ">90"]

df_multi["rating_group"] = pd.cut(
    df_multi["positive_rate"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_multi = df_multi.dropna(subset=["rating_group"]).copy()

df_multi["rating_group"].value_counts().sort_index()

In [ ]:
le = LabelEncoder()

df_multi["rating_group_encoded"] = le.fit_transform(df_multi["rating_group"])

class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
class_mapping

## 6. Feature Selection

Columns that directly leak the target or are identifiers are removed.  
The remaining features represent game metadata, pricing, platform support, and categorical information.

In [ ]:
drop_cols = [
    # Target and leakage columns
    "rating_group",
    "rating_group_encoded",
    "is_highly_rated",
    "positive_rate",
    "positive_reviews",
    "negative_reviews",
    "total_reviews",
    "total_reviews_clean",
    "positive_reviews_clean",
    "negative_reviews_clean",

    # Review behavior columns that would not be available before reviews
    "avg_playtime_forever",
    "avg_playtime_at_review",
    "avg_weighted_vote_score",
    "total_votes_up",
    "total_votes_funny",
    "steam_purchase_reviews",
    "received_for_free_reviews",
    "early_access_reviews",

    # Identifiers / non-modeling columns
    "appid",
    "name",
    "release_date"
]

X_multi = df_multi.drop(columns=[col for col in drop_cols if col in df_multi.columns])
y_multi = df_multi["rating_group_encoded"]

print("Feature matrix shape:", X_multi.shape)
print("Target shape:", y_multi.shape)

In [ ]:
X_multi.dtypes

## 7. Train-Test Split

A stratified split is used to preserve the class distribution in both training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_multi,
    y_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_multi
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True).sort_index())

## 8. Missing Value Handling and Categorical Conversion

Numerical missing values are filled using simple train-based rules.  
Categorical columns are converted to the `category` dtype for LightGBM.

In [ ]:
# Numeric imputations
if "metacritic_score" in X_train.columns:
    X_train["metacritic_score"] = X_train["metacritic_score"].fillna(0)
    X_test["metacritic_score"] = X_test["metacritic_score"].fillna(0)

if "release_year" in X_train.columns:
    median_year = X_train["release_year"].median()
    X_train["release_year"] = X_train["release_year"].fillna(median_year)
    X_test["release_year"] = X_test["release_year"].fillna(median_year)

if "price_usd" in X_train.columns:
    median_price = X_train["price_usd"].median()
    X_train["price_usd"] = X_train["price_usd"].fillna(median_price)
    X_test["price_usd"] = X_test["price_usd"].fillna(median_price)

# Categorical conversion
cat_cols = [
    "is_free",
    "mat_supports_windows",
    "mat_supports_mac",
    "mat_supports_linux",
    "price_type",
    "type_clean",
    "genres_clean",
    "developers_clean",
    "publishers_clean",
    "categories_clean",
    "has_metacritic"
]

for col in cat_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

X_train.info()

## 9. Baseline LightGBM Multiclass Model

The first model is a baseline LightGBM multiclass classifier.

In [ ]:
lgbm_base = LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    metric="multi_logloss",
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1
)

problematic_object_cols = ['type', 'genres', 'developers', 'publishers', 'categories']
for col in problematic_object_cols:
    if col in X_train.columns:
        X_train = X_train.drop(columns=[col])
    if col in X_test.columns:
        X_test = X_test.drop(columns=[col])

lgbm_base.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)

## 10. Baseline Model Evaluation

The model is evaluated using accuracy, weighted precision, weighted recall, weighted F1-score, and a classification report.

In [ ]:
y_pred_base = lgbm_base.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_base))
print("Precision:", precision_score(y_test, y_pred_base, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_base, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_base, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_base, target_names=le.classes_))

In [ ]:
cm_base = confusion_matrix(y_test, y_pred_base)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_base,
    display_labels=le.classes_
)

disp.plot(values_format="d", xticks_rotation=45,  cmap='Blues')
plt.title("Baseline LightGBM - Confusion Matrix")
plt.show()

## 11. LightGBM with Class Weights

Because rating groups may be imbalanced, class weights are calculated and passed to LightGBM.

In [ ]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
class_weights

In [ ]:
lgbm_weighted = LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    class_weight=class_weights,
    metric="multi_logloss",
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1
)

lgbm_weighted.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)

## 12. Weighted Model Evaluation

In [ ]:
y_pred_weighted = lgbm_weighted.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_weighted))
print("Precision:", precision_score(y_test, y_pred_weighted, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_weighted, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_weighted, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_weighted, target_names=le.classes_))

In [ ]:
cm_weighted = confusion_matrix(y_test, y_pred_weighted)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_weighted,
    display_labels=le.classes_
)

disp.plot(values_format="d", xticks_rotation=45,  cmap='Blues')
plt.title("Weighted LightGBM - Confusion Matrix")
plt.show()

## 13. Feature Importance

Feature importance helps identify which variables contribute most to the multiclass classification model.

In [ ]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": lgbm_weighted.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(15)

In [ ]:
top_features = importance_df.head(15).sort_values("importance")

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top 15 Feature Importances - Weighted LightGBM")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 14. Optional: CatBoost Multiclass Model

CatBoost can handle categorical variables effectively.  
This section is optional and may take longer to run.

In [ ]:
# Install CatBoost in Colab if needed
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier

for col in ['developers_clean', 'publishers_clean']:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype(object).fillna("Unknown").astype("category")
        X_test[col] = X_test[col].astype(object).fillna("Unknown").astype("category")

cat_feature_indices = [
    X_train.columns.get_loc(col)
    for col in cat_cols
    if col in X_train.columns
]

cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_test, y_test),
    use_best_model=True
)

## 15. CatBoost Evaluation

In [ ]:
y_pred_cat = cat_model.predict(X_test).flatten()

print("Accuracy :", accuracy_score(y_test, y_pred_cat))
print("Precision:", precision_score(y_test, y_pred_cat, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_cat, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_cat, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat, target_names=le.classes_))

In [ ]:
cm_cat = confusion_matrix(y_test, y_pred_cat)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_cat,
    display_labels=le.classes_
)

disp.plot(values_format="d", xticks_rotation=45,  cmap='Blues')
plt.title("CatBoost Multiclass - Confusion Matrix")
plt.show()

In [ ]:
catboost_importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

catboost_importance_df.head(15)

## 16. Optimized CatBoost Multiclass Model with Optuna

This section tunes CatBoost hyperparameters using Optuna.  
The optimization objective is weighted F1-score, which is more suitable than accuracy when class distributions are imbalanced.

> Note: This section may take longer to run. You can reduce `n_trials` if needed.


In [ ]:
from sklearn.model_selection import train_test_split

X_train_opt, X_valid, y_train_opt, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
# Install Optuna in Colab if needed
!pip -q install optuna

import optuna
from sklearn.metrics import f1_score


def objective_catboost_multiclass(trial):
    params = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        "iterations": trial.suggest_int("iterations", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "random_seed": 42,
        "verbose": 0,
        "auto_class_weights": "Balanced",
        "early_stopping_rounds": 50
    }

    model = CatBoostClassifier(**params)

    model.fit(
        X_train_opt,
        y_train_opt,
        cat_features=cat_feature_indices,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    y_pred_valid = model.predict(X_valid).flatten()
    return f1_score(y_valid, y_pred_valid, average="weighted")


In [ ]:
study_catboost_multi = optuna.create_study(
    direction="maximize",
    study_name="CatBoost_Multiclass_Optimization"
)

study_catboost_multi.optimize(
    objective_catboost_multiclass,
    n_trials=30,
    show_progress_bar=True
)

print("Best weighted F1:", study_catboost_multi.best_value)
print("Best parameters:")
for key, value in study_catboost_multi.best_params.items():
    print(f"{key}: {value}")


## 17. Retrain Optimized CatBoost Model

After Optuna finds the best hyperparameters, the final CatBoost model is retrained using those parameters.


In [ ]:
best_params_catboost_multi = study_catboost_multi.best_params

best_catboost_model_multi = CatBoostClassifier(
    **best_params_catboost_multi,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

best_catboost_model_multi.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_test, y_test),
    use_best_model=True
)

print("Optimized CatBoost multiclass model trained successfully.")


## 18. Optimized CatBoost Evaluation

The optimized CatBoost model is evaluated with the same metrics used for the previous models.


In [ ]:
y_pred_cat_opt = best_catboost_model_multi.predict(X_test).flatten()

print("Accuracy :", accuracy_score(y_test, y_pred_cat_opt))
print("Precision:", precision_score(y_test, y_pred_cat_opt, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_cat_opt, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_cat_opt, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat_opt, target_names=le.classes_))

In [ ]:
cm_cat_opt = confusion_matrix(y_test, y_pred_cat_opt)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_cat_opt,
    display_labels=le.classes_
)

disp.plot(values_format="d", xticks_rotation=45, cmap='Blues')
plt.title("Optimized CatBoost Multiclass - Confusion Matrix")
plt.show()


In [ ]:
catboost_opt_importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_catboost_model_multi.get_feature_importance()
}).sort_values("importance", ascending=False)

catboost_opt_importance_df.head(15)


In [ ]:
top_catboost_features = catboost_opt_importance_df.head(15).sort_values("importance")

plt.figure(figsize=(10, 6))
plt.barh(top_catboost_features["feature"], top_catboost_features["importance"])
plt.title("Top 15 Feature Importances - Optimized CatBoost Multiclass")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 19. Model Comparison

This section summarizes the main performance metrics of all multiclass models.


In [ ]:
results = pd.DataFrame([
    {
        "model": "LightGBM Baseline",
        "accuracy": accuracy_score(y_test, y_pred_base),
        "precision_weighted": precision_score(y_test, y_pred_base, average="weighted"),
        "recall_weighted": recall_score(y_test, y_pred_base, average="weighted"),
        "f1_weighted": f1_score(y_test, y_pred_base, average="weighted")
    },
    {
        "model": "LightGBM Weighted",
        "accuracy": accuracy_score(y_test, y_pred_weighted),
        "precision_weighted": precision_score(y_test, y_pred_weighted, average="weighted"),
        "recall_weighted": recall_score(y_test, y_pred_weighted, average="weighted"),
        "f1_weighted": f1_score(y_test, y_pred_weighted, average="weighted")
    },
    {
        "model": "CatBoost Multiclass",
        "accuracy": accuracy_score(y_test, y_pred_cat),
        "precision_weighted": precision_score(y_test, y_pred_cat, average="weighted"),
        "recall_weighted": recall_score(y_test, y_pred_cat, average="weighted"),
        "f1_weighted": f1_score(y_test, y_pred_cat, average="weighted")
    },
    {
        "model": "Optimized CatBoost Multiclass",
        "accuracy": accuracy_score(y_test, y_pred_cat_opt),
        "precision_weighted": precision_score(y_test, y_pred_cat_opt, average="weighted"),
        "recall_weighted": recall_score(y_test, y_pred_cat_opt, average="weighted"),
        "f1_weighted": f1_score(y_test, y_pred_cat_opt, average="weighted")
    }
])

results.sort_values("f1_weighted", ascending=False)


In [ ]:
print(study_catboost_multi.best_value)

print(study_catboost_multi.best_params)

## 20. Conclusion

This notebook explored multiclass classification for Steam game rating groups.

Key points:

- The target was created from `positive_rate` intervals.
- Target leakage columns such as `positive_rate`, review counts, and review-based metrics were excluded from the feature matrix.
- LightGBM was used as the baseline multiclass model.
- A weighted LightGBM model was tested to address class imbalance.
- CatBoost was added because it can handle categorical variables effectively.
- Optimized CatBoost was tuned with Optuna and compared against the baseline models.

The final model choice should be based mainly on weighted F1-score and class-level performance in the classification report.
